# 11 — Profiler trace analysis: breakdown, timeline, trace_id joins

**Learning goal.** Open a generated Chrome-trace JSON file, compute the time breakdown (compile / device / collective / input_pipeline / host / other), and plot a horizontal-bar timeline of events. Then show how a single `trace_id` connects log lines, metric rows, and trace events end to end.

**What you'll observe.**
- A loaded trace from `../artifacts/traces/run_*.json`.
- A `Breakdown` from `compute_breakdown(...)` and a small horizontal-bar timeline.
- A walk-through of joining the same `trace_id` across the log JSONL, metric CSV, and trace JSON.

**Knobs to tune.**
- Which trace file to load (most recent by default).
- The plot's max number of events (long traces clutter the view).

**Failure modes to watch.**
- Chrome traces store time as **microseconds**; convert if mixing with seconds.
- The trace JSON the demo writes uses Chrome's `traceEvents` array — load it with `json.load` and read the `["traceEvents"]` key.

In [ ]:
import sys, os
from pathlib import Path
# Add the parent of cloud_tpu_lab to sys.path so `cloud_tpu_lab.src.*` imports work.
HERE = Path(os.getcwd()).resolve()
_root = HERE
for _ in range(5):
    if (_root / "cloud_tpu_lab").exists():
        break
    _root = _root.parent
sys.path.insert(0, str(_root))
sys.path.insert(0, "..")
PLOT_DIR = Path("../artifacts/plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)
print("repo root:", _root)
print("plot dir:", PLOT_DIR.resolve())

## Locate the most recent trace

If you've never run the demo, the cell below regenerates one quickly.

In [ ]:
TRACE_DIR = Path("../artifacts/traces").resolve()
trace_files = sorted(TRACE_DIR.glob("run_*.json"))

if not trace_files:
    print("No traces found — generating one quickly...")
    from cloud_tpu_lab.examples.run_cpu_simulation_demo import run_demo
    from cloud_tpu_lab.src.common.config import WorkloadConfig, SimulationConfig
    run_demo(WorkloadConfig(name="nb11_gen", n_steps=4), SimulationConfig(), quiet=True)
    trace_files = sorted(TRACE_DIR.glob("run_*.json"))

trace_path = trace_files[-1]
print("Loading:", trace_path)

## Load the trace and rebuild a ProfilerTrace

In [ ]:
import json

with open(trace_path) as f:
    chrome = json.load(f)

events = chrome["traceEvents"]
trace_id = chrome.get("metadata", {}).get("trace_id")
print("trace_id:", trace_id)
print("n events:", len(events))

# Show a couple of events
for e in events[:3]:
    print(e)

## Reconstruct a ProfilerTrace and compute the breakdown

In [ ]:
from cloud_tpu_lab.src.profiling.profiler_trace import ProfilerTrace, TraceEvent
from cloud_tpu_lab.src.profiling.trace_analyzer import compute_breakdown, step_summary

pt = ProfilerTrace(trace_id=trace_id or "unknown")
for e in events:
    pt.events.append(TraceEvent(
        name=e["name"], cat=e["cat"],
        ts_us=e["ts"], dur_us=e["dur"],
        tid=e.get("tid", 0), args=e.get("args", {}),
    ))

breakdown = compute_breakdown(pt)
print("breakdown (seconds):")
for c in ("compile", "device", "collective", "input_pipeline", "host", "checkpoint"):
    print(f"  {c:<15} {getattr(breakdown, c + '_s'):.6f}")
print(f"  {'other':<15} {breakdown.other_s:.6f}")
print(f"  {'TOTAL':<15} {breakdown.total():.6f}")

print("\nfractions:")
for k, v in breakdown.fractions().items():
    print(f"  {k:<15} {v*100:5.1f}%")

## Timeline: horizontal bars by category

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

CAT_TID = {"compile": 0, "input_pipeline": 1, "device": 2, "collective": 3}

# Limit to first ~60 events to keep the plot readable.
events_to_plot = events[:60]
fig = plt.figure(figsize=(10, 4))
ax = plt.gca()
for e in events_to_plot:
    tid = CAT_TID.get(e["cat"], 4)
    ax.barh(tid, width=e["dur"] / 1000, left=e["ts"] / 1000,
            height=0.6, edgecolor="black", linewidth=0.4)

ax.set_yticks(list(CAT_TID.values()) + [4])
ax.set_yticklabels(list(CAT_TID.keys()) + ["other"])
ax.set_xlabel("time (ms from trace start)")
ax.set_title(f"Profiler timeline — {trace_id}  (first {len(events_to_plot)} events)")
ax.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
out_png = PLOT_DIR / f"nb11_timeline_{trace_id or 'unknown'}.png"
plt.savefig(out_png, dpi=120)
plt.show()
print("saved:", out_png)

## Finding one trace_id end to end

Every artefact in this lab is keyed on `trace_id`. To follow a single run:

- **Log** lines live at `../artifacts/logs/run_<trace_id>.jsonl` (one event per line).
- **Metrics** rows live at `../artifacts/metrics/run_<trace_id>.csv`.
- **Trace** events live at `../artifacts/traces/run_<trace_id>.json` (the one we just loaded).

Open all three by the same `trace_id` and you get the full picture: what was logged, what was measured, and when on a wall clock.

In [ ]:
tid = trace_id or "TRACE-0001"
log_path = Path(f"../artifacts/logs/run_{tid}.jsonl").resolve()
metrics_path = Path(f"../artifacts/metrics/run_{tid}.csv").resolve()
print("log    :", log_path, "exists:", log_path.exists())
print("metrics:", metrics_path, "exists:", metrics_path.exists())
print("trace  :", trace_path, "exists:", trace_path.exists())

# Print first 3 log lines for this trace_id.
if log_path.exists():
    with open(log_path) as f:
        for i, line in enumerate(f):
            if i >= 3:
                break
            print("LOG:", line.strip()[:200])

## Exercises

1. Run the demo with `chip_count=4` and replot the timeline. The `collective` row should now be non-empty.
2. From the loaded trace, extract every event with `cat="device"` and `name.startswith("step")`. Compute p50, p95, and max step time.
3. Use `step_summary(pt)` and compare its `p95_step_s` to what you computed by hand. Should match.
4. Filter `events` by a specific `args["trace_id"]` (most events carry it). What's the maximum span (last_event_end - first_event_start)? Is that the wall-clock for the run?
5. Pick one `step_id` and produce the joined record: log lines + metric row + trace events for that step only. How many events did that step generate?